# Pennsylvania Severe Weather Atlas
## Notebook 06 — Atlas Summary and Final Review

### Purpose

Bring together the findings from Notebooks 1–5 and prepare a final atlas summary supported by the saved datasets, tables, and figures.

### Study scope

- **Period:** 2011–2025.
- **Statewide view:** Pennsylvania.
- **Local view:** Lehigh and Northampton counties.
- **Event types:** Thunderstorm Wind, Hail, and Tornado.

### Notebook goals

1. Load the final dataset and supporting summaries.
2. Confirm that the saved outputs support our key findings.
3. Summarize annual, seasonal, geographic, magnitude, and impact patterns.
4. Document interpretation limits and organize the final deliverables.

### Imports and environment check

The next cell imports tools for reading tables, checking numerical results, and displaying saved figures. It also identifies the Python environment running this notebook.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from IPython.display import Image, Markdown, display

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Virtual environment:", sys.prefix != sys.base_prefix)
print("Notebook 6 imports loaded successfully.")

Python: 3.13.5
Interpreter: c:\Users\rcolo\OneDrive\Desktop\Pennsylvania-Severe-Weather-Atlas\.venv\Scripts\python.exe
Virtual environment: True
Notebook 6 imports loaded successfully.


## Loading the final event dataset

We will load the enriched dataset saved in Notebook 5.

County identifiers will remain text, flags will retain their Boolean types, and the numeric damage columns will preserve missing estimates.

The initial overview will show the dataset’s size, study years, county coverage, and event counts.

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_PATHS = {
    "processed": PROJECT_ROOT / "data" / "processed",
    "tables": PROJECT_ROOT / "reports" / "tables",
    "figures": PROJECT_ROOT / "reports" / "figures",
    "maps": PROJECT_ROOT / "outputs" / "maps",
    "reports": PROJECT_ROOT / "reports",
}

FOCUS_EVENT_TYPES = ["Thunderstorm Wind", "Hail", "Tornado"]

dataset_path = (
    PROJECT_PATHS["processed"]
    / "pa_focus_events_with_impacts_2011_2025.csv"
)

atlas_events = pd.read_csv(
    dataset_path,
    dtype={
        "county_fips": "string",
        "is_lehigh_valley": "boolean",
        "hail_ge_1in": "boolean",
        "wind_ge_50kt": "boolean",
        "DAMAGE_PROPERTY": "string",
        "DAMAGE_CROPS": "string",
        "property_damage_usd": "Float64",
        "crop_damage_usd": "Float64",
    },
    parse_dates=["begin_datetime"],
    low_memory=False,
)

print("Project root:", PROJECT_ROOT)
print(f"Rows: {len(atlas_events):,}")
print("Columns:", atlas_events.shape[1])
print(
    "Year range:",
    atlas_events["event_year"].min(),
    "to",
    atlas_events["event_year"].max(),
)
print("Distinct counties:", atlas_events["county_fips"].nunique())
print(f"Lehigh Valley records: {atlas_events['is_lehigh_valley'].sum():,}")

final_event_counts = (
    atlas_events["EVENT_TYPE"]
    .value_counts()
    .reindex(FOCUS_EVENT_TYPES, fill_value=0)
    .rename_axis("event_type")
    .reset_index(name="record_count")
)

final_event_counts

Project root: c:\Users\rcolo\OneDrive\Desktop\Pennsylvania-Severe-Weather-Atlas
Rows: 15,399
Columns: 62
Year range: 2011 to 2025
Distinct counties: 67
Lehigh Valley records: 564


,event_type,record_count
0,Thunderstorm Wind,12630
1,Hail,2441
2,Tornado,328


## Final dataset overview

The enriched dataset contains **15,399 records and 62 columns**, covering 2011–2025 and 67 counties. The Lehigh Valley subset contains **564 records**.

The event counts match our earlier summaries: 12,630 Thunderstorm Wind records, 2,441 Hail records, and 328 Tornado records.

## Loading supporting summaries

We will load the saved annual, monthly, seasonal, county, magnitude, and impact summaries, along with the source registry.

These exported files will support the final findings. After loading them, we will check their consistency with the final event dataset.

In [3]:
atlas_table_files = {
    "pa_annual": "pa_annual_focus_counts_2011_2025.csv",
    "pa_monthly": "pa_monthly_focus_counts_2011_2025.csv",
    "pa_seasonal": "pa_seasonal_focus_counts_2011_2025.csv",
    "pa_county": "pa_county_focus_counts_2011_2025.csv",
    "lv_annual": "lehigh_valley_annual_focus_counts_2011_2025.csv",
    "lv_monthly": "lehigh_valley_monthly_focus_counts_2011_2025.csv",
    "lv_seasonal": "lehigh_valley_seasonal_focus_counts_2011_2025.csv",
    "lv_county": "lehigh_valley_county_focus_counts_2011_2025.csv",
    "hail_sizes": "pa_lehigh_valley_hail_sizes_2011_2025.csv",
    "wind_magnitudes": "pa_lehigh_valley_wind_magnitudes_2011_2025.csv",
    "tornado_ratings": "pa_lehigh_valley_tornado_ratings_2011_2025.csv",
    "injuries_fatalities": "pa_lehigh_valley_injuries_fatalities_2011_2025.csv",
    "reported_damage": "pa_lehigh_valley_reported_damage_2011_2025.csv",
    "source_registry": "data_source_registry.csv",
}

atlas_tables = {}
loaded_table_rows = []

for name, filename in atlas_table_files.items():
    table_path = PROJECT_PATHS["tables"] / filename
    csv_dtypes = {"county_fips": "string"} if name.endswith("_county") else None

    table = pd.read_csv(table_path, dtype=csv_dtypes)
    atlas_tables[name] = table

    loaded_table_rows.append({
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
    })

loaded_table_inventory = pd.DataFrame(loaded_table_rows)

print("Supporting tables loaded:", len(atlas_tables))
loaded_table_inventory

Supporting tables loaded: 14


,table,rows,columns
0,pa_annual,15,5
1,pa_monthly,12,5
2,pa_seasonal,4,6
3,pa_county,67,7
4,lv_annual,15,5
5,lv_monthly,12,5
6,lv_seasonal,4,6
7,lv_county,2,6
8,hail_sizes,13,5
9,wind_magnitudes,4,7


## Supporting tables loaded

All 14 supporting tables loaded successfully.

The annual summaries contain 15 rows each, monthly summaries contain 12, and seasonal summaries contain four. The county summaries cover 67 Pennsylvania counties and the two Lehigh Valley counties.

The magnitude summaries, impact summaries, and two-source registry are also available for the final review.

## Checking time and county summaries

We will rebuild the annual, monthly, seasonal, and county counts from the final dataset.

For each saved table, we will check:

- Whether the group labels match.
- Whether each group’s event counts and row total match.
- Whether the overall total accounts for every record in the relevant region.

Monthly tables include zero-count months. Seasonal grouping follows the earlier analysis: December–February, March–May, June–August, and September–November.

In [4]:
MONTH_LABELS = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]

SEASON_BY_MONTH = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

review_months = atlas_events["begin_datetime"].dt.month

review_events = atlas_events.assign(
    year=atlas_events["begin_datetime"].dt.year,
    month=review_months.map(dict(enumerate(MONTH_LABELS, start=1))),
    season=review_months.map(SEASON_BY_MONTH),
)


def check_count_summary(table_name, records, group_column, groups):
    expected = pd.crosstab(
        records[group_column], records["EVENT_TYPE"]
    ).reindex(
        index=groups, columns=FOCUS_EVENT_TYPES, fill_value=0
    )
    expected["total_records"] = expected.sum(axis=1)

    saved = atlas_tables[table_name].set_index(group_column)

    groups_match = (
        saved.index.is_unique
        and set(saved.index) == set(expected.index)
    )
    counts_match = groups_match and np.array_equal(
        saved.reindex(expected.index)[expected.columns].to_numpy(),
        expected.to_numpy(),
    )

    return {
        "table": table_name,
        "groups_match": groups_match,
        "counts_match": counts_match,
        "record_total_matches": bool(
            saved["total_records"].sum() == len(records)
            and expected["total_records"].sum() == len(records)
        ),
    }


count_check_rows = []

for prefix, records in [
    ("pa", review_events),
    ("lv", review_events.loc[review_events["is_lehigh_valley"]]),
]:
    group_specs = [
        ("annual", "year", range(2011, 2026)),
        ("monthly", "month", MONTH_LABELS),
        ("seasonal", "season", ["Winter", "Spring", "Summer", "Autumn"]),
        ("county", "county_fips", records["county_fips"].dropna().unique()),
    ]

    for suffix, column, groups in group_specs:
        count_check_rows.append(
            check_count_summary(
                f"{prefix}_{suffix}", records, column, groups
            )
        )

final_count_checks = pd.DataFrame(count_check_rows)

final_count_checks

,table,groups_match,counts_match,record_total_matches
0,pa_annual,True,True,True
1,pa_monthly,False,False,True
2,pa_seasonal,True,True,True
3,pa_county,True,True,True
4,lv_annual,True,True,True
5,lv_monthly,False,False,True
6,lv_seasonal,True,True,True
7,lv_county,True,True,True


## Interpreting the summary checks

The Pennsylvania and Lehigh Valley annual, seasonal, and county tables match the final dataset on group labels, counts, and record totals.

Both monthly tables have the correct overall record totals. However, their month labels differ from the labels expected by the verification code.

The monthly `counts_match=False` results do not yet establish a count discrepancy: that comparison only runs when the group labels match.

Next, inspect the saved month labels and their data types.

In [5]:
print("Expected month labels:", MONTH_LABELS)

for table_name in ["pa_monthly", "lv_monthly"]:
    saved_months = atlas_tables[table_name]["month"]

    print(f"\n{table_name}")
    print("Month column type:", saved_months.dtype)
    print("Saved month labels:", [repr(value) for value in saved_months])
    print("Missing month labels:", saved_months.isna().sum())
    print("Repeated month labels:", saved_months.duplicated().sum())

Expected month labels: ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

pa_monthly
Month column type: int64
Saved month labels: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
Missing month labels: 0
Repeated month labels: 0

lv_monthly
Month column type: int64
Saved month labels: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
Missing month labels: 0
Repeated month labels: 0


## Resolving the monthly label mismatch

Both monthly tables contain all 12 numeric month identifiers, with no missing or repeated labels.

The initial verification expected abbreviated month names, causing the group checks to fail. We will use numeric months on both sides of the comparison and rerun the monthly count checks.

In [6]:
monthly_review_events = atlas_events.assign(
    month=atlas_events["begin_datetime"].dt.month
)

for prefix, records in [
    ("pa", monthly_review_events),
    ("lv", monthly_review_events.loc[monthly_review_events["is_lehigh_valley"]]),
]:
    monthly_result = check_count_summary(
        f"{prefix}_monthly", records, "month", range(1, 13)
    )

    matching_row = final_count_checks["table"].eq(monthly_result["table"])

    for column, value in monthly_result.items():
        final_count_checks.loc[matching_row, column] = value

final_count_checks

,table,groups_match,counts_match,record_total_matches
0,pa_annual,True,True,True
1,pa_monthly,True,True,True
2,pa_seasonal,True,True,True
3,pa_county,True,True,True
4,lv_annual,True,True,True
5,lv_monthly,True,True,True
6,lv_seasonal,True,True,True
7,lv_county,True,True,True


## Interpreting the completed count checks

All eight annual, monthly, seasonal, and county tables match the final dataset on group identifiers, event-type counts, and record totals.

The earlier monthly failures came from comparing numeric month identifiers with month names. Using numeric identifiers on both sides resolved the mismatch.

## Building the final atlas overview

Create a compact comparison of Pennsylvania and the Lehigh Valley: record totals, event types, county coverage, and the years, months, and counties with the most records.

Monthly totals combine the same calendar month across 2011–2025. Summer includes June, July, and August. The Lehigh Valley is included in the Pennsylvania totals.

In [7]:
def describe_peak(table, group_column, label_map=None):
    largest_count = int(table["total_records"].max())

    labels = table.loc[
        table["total_records"].eq(largest_count), group_column
    ]

    if label_map is not None:
        labels = labels.map(label_map)

    names = ", ".join(labels.astype(str))
    return f"{names} ({largest_count:,} records)"


month_labels_by_number = dict(enumerate(MONTH_LABELS, start=1))
overview_columns = {}

for prefix, region in [
    ("pa", "Pennsylvania"),
    ("lv", "Lehigh Valley"),
]:
    annual = atlas_tables[f"{prefix}_annual"]
    monthly = atlas_tables[f"{prefix}_monthly"]
    seasonal = atlas_tables[f"{prefix}_seasonal"]
    county = atlas_tables[f"{prefix}_county"]

    total_records = int(annual["total_records"].sum())
    event_totals = annual[FOCUS_EVENT_TYPES].sum()

    summer_records = int(
        seasonal.loc[
            seasonal["season"].eq("Summer"), "total_records"
        ].iloc[0]
    )

    overview_columns[region] = {
        "Reported event records": f"{total_records:,}",
        "Thunderstorm Wind records": f"{event_totals['Thunderstorm Wind']:,}",
        "Hail records": f"{event_totals['Hail']:,}",
        "Tornado records": f"{event_totals['Tornado']:,}",
        "Counties represented": str(len(county)),
        "Year with most records": describe_peak(annual, "year"),
        "Month with most pooled records": describe_peak(
            monthly, "month", month_labels_by_number
        ),
        "Summer records (share of region total)": (
            f"{summer_records:,} ({summer_records / total_records:.2%})"
        ),
        "County with most records": describe_peak(county, "county_name"),
    }

atlas_overview = (
    pd.DataFrame(overview_columns)
    .rename_axis("measure")
    .reset_index()
)

atlas_overview

,measure,Pennsylvania,Lehigh Valley
0,Reported event records,"15,399",564
1,Thunderstorm Wind records,"12,630",464
2,Hail records,"2,441",93
3,Tornado records,328,7
4,Counties represented,67,2
5,Year with most records,"2019 (1,571 records)",2020 (98 records)
6,Month with most pooled records,"Jul (3,528 records)",Jul (165 records)
7,Summer records (share of region total),"9,342 (60.67%)",391 (69.33%)
8,County with most records,"Allegheny (1,063 records)",Lehigh (313 records)


## Interpreting the atlas overview

Thunderstorm Wind accounts for the largest number of records in both regions.

Pennsylvania’s highest annual total was **1,571 records in 2019**; the Lehigh Valley’s was **98 in 2020**.

July has the most pooled monthly records in both regions. Summer accounts for **60.67%** of Pennsylvania records and **69.33%** of Lehigh Valley records.

Allegheny leads Pennsylvania’s county totals with **1,063 records**. Within the Lehigh Valley, Lehigh has **313 records**, compared with Northampton’s **251**.

## Consolidating reported impacts

Create a regional comparison of reported injuries, fatalities, and damage estimates. Keep direct and indirect impacts separate.

Dollar totals combine available estimates across 2011–2025 without inflation adjustment. Include missing estimates and explicit crop-damage zeros so that unknown amounts remain distinguishable from reported zeros.

In [9]:
def format_reported_usd(values):
    total = values.sum(min_count=1)
    return "No available estimates" if pd.isna(total) else f"${total:,.0f}"


impact_overview_columns = {}

for region, records in [
    ("Pennsylvania", atlas_events),
    ("Lehigh Valley", atlas_events.loc[atlas_events["is_lehigh_valley"]]),
]:
    property_estimates = records["property_damage_usd"]
    crop_estimates = records["crop_damage_usd"]

    impact_overview_columns[region] = {
        "Reported direct injuries": f"{records['INJURIES_DIRECT'].sum():,}",
        "Reported indirect injuries": f"{records['INJURIES_INDIRECT'].sum():,}",
        "Reported direct fatalities": f"{records['DEATHS_DIRECT'].sum():,}",
        "Reported indirect fatalities": f"{records['DEATHS_INDIRECT'].sum():,}",
        "Available property damage total (USD)": format_reported_usd(
            property_estimates
        ),
        "Missing property damage estimates": f"{property_estimates.isna().sum():,}",
        "Available crop damage total (USD)": format_reported_usd(
            crop_estimates
        ),
        "Missing crop damage estimates": f"{crop_estimates.isna().sum():,}",
        "Explicit zero crop damage estimates": f"{crop_estimates.eq(0).sum():,}",
        "Positive crop damage estimates": f"{crop_estimates.gt(0).sum():,}",
    }

atlas_impact_overview = (
    pd.DataFrame(impact_overview_columns)
    .rename_axis("measure")
    .reset_index()
)

atlas_impact_overview

,measure,Pennsylvania,Lehigh Valley
0,Reported direct injuries,205,2
1,Reported indirect injuries,10,0
2,Reported direct fatalities,18,0
3,Reported indirect fatalities,5,0
4,Available property damage total (USD),"$203,508,160","$1,691,000"
5,Missing property damage estimates,"2,127",164
6,Available crop damage total (USD),"$2,156,000",$0
7,Missing crop damage estimates,"2,248",166
8,Explicit zero crop damage estimates,"13,010",398
9,Positive crop damage estimates,141,0


## Interpreting the reported impact overview

Pennsylvania’s focus records report **205 direct and 10 indirect injuries**, alongside **18 direct and 5 indirect fatalities**.

Available estimates total **$203,508,160 in property damage** and **$2,156,000 in crop damage**. These sums exclude **2,127 missing property estimates** and **2,248 missing crop estimates**.

Lehigh Valley records report **2 direct injuries and no fatalities**. Available property estimates total **$1,691,000**, with **164 property estimates missing**.

The Lehigh Valley’s available crop-damage total is **$0**, comprising **398 explicit zero estimates** and no positive estimates. Another **166 estimates are missing**, so this does not establish that no crop damage occurred.

## Organizing the atlas visuals

Inventory the charts and maps created in Notebooks 3–5. Record their locations relative to the project folder and check that the files exist.

This inventory will supply figure links for the final atlas report.

In [10]:
atlas_visual_files = {
    "Pennsylvania annual patterns": (
        "figures", "pa_annual_focus_events_2011_2025.png"
    ),
    "Pennsylvania monthly heatmap": (
        "figures", "pa_monthly_focus_share_heatmap_2011_2025.png"
    ),
    "Lehigh Valley annual patterns": (
        "figures", "lehigh_valley_annual_focus_events_2011_2025.png"
    ),
    "Statewide and local monthly comparison": (
        "figures", "pa_lehigh_valley_monthly_comparison_2011_2025.png"
    ),
    "County total records map": (
        "maps", "pa_county_total_records_2011_2025.png"
    ),
    "County maps by event type": (
        "maps", "pa_county_records_by_event_type_2011_2025.png"
    ),
    "Hail size distributions": (
        "figures", "pa_lehigh_valley_hail_size_distribution_2011_2025.png"
    ),
    "Wind magnitude distributions": (
        "figures", "pa_lehigh_valley_wind_ecdf_2011_2025.png"
    ),
    "Tornado ratings": (
        "figures", "pa_lehigh_valley_tornado_ratings_2011_2025.png"
    ),
    "Pennsylvania injuries and fatalities": (
        "figures", "pa_reported_injuries_fatalities_2011_2025.png"
    ),
    "Property and crop damage": (
        "figures", "pa_lehigh_valley_reported_damage_2011_2025.png"
    ),
}

visual_inventory_rows = []

for title, (folder, filename) in atlas_visual_files.items():
    visual_path = PROJECT_PATHS[folder] / filename
    exists = visual_path.is_file()

    visual_inventory_rows.append({
        "visual": title,
        "relative_path": visual_path.relative_to(PROJECT_ROOT).as_posix(),
        "file_exists": exists,
        "file_size_bytes": visual_path.stat().st_size if exists else None,
    })

atlas_visual_inventory = pd.DataFrame(visual_inventory_rows)

print(
    "Visual files found:",
    atlas_visual_inventory["file_exists"].sum(),
    "of",
    len(atlas_visual_inventory),
)

atlas_visual_inventory[["visual", "file_exists", "file_size_bytes"]]

Visual files found: 11 of 11


,visual,file_exists,file_size_bytes
0,Pennsylvania annual patterns,True,344739
1,Pennsylvania monthly heatmap,True,204048
2,Lehigh Valley annual patterns,True,166557
3,Statewide and local monthly comparison,True,453384
4,County total records map,True,434213
5,County maps by event type,True,781299
6,Hail size distributions,True,246280
7,Wind magnitude distributions,True,231337
8,Tornado ratings,True,195820
9,Pennsylvania injuries and fatalities,True,156726


## Interpreting the visual inventory

All 11 expected atlas visuals were found successfully.

The deliverables cover statewide and Lehigh Valley time patterns, county geography, hail sizes, wind magnitudes, tornado ratings, reported impacts, and estimated damage.

## Saving the final review outputs

Save the overview tables and visual inventory in `reports/tables` so the completed atlas has reusable summary files alongside its figures and maps.

In [11]:
final_review_specs = {
    "atlas_overview": (
        atlas_overview,
        "atlas_overview_2011_2025.csv",
    ),
    "atlas_impact_overview": (
        atlas_impact_overview,
        "atlas_impact_overview_2011_2025.csv",
    ),
    "atlas_visual_inventory": (
        atlas_visual_inventory,
        "atlas_visual_inventory.csv",
    ),
}

final_review_rows = []

for table_name, (table, filename) in final_review_specs.items():
    output_path = PROJECT_PATHS["tables"] / filename
    table.to_csv(output_path, index=False)

    reloaded = pd.read_csv(output_path)

    pd.testing.assert_frame_equal(
        table.reset_index(drop=True),
        reloaded.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-8,
        obj=filename,
    )

    final_review_rows.append({
        "table": table_name,
        "file": filename,
        "rows": len(table),
        "columns": table.shape[1],
        "file_exists": output_path.is_file(),
        "values_match_after_reload": True,
    })

final_review_inventory = pd.DataFrame(final_review_rows)

manifest_path = PROJECT_PATHS["tables"] / "atlas_final_review_manifest.csv"
final_review_inventory.to_csv(manifest_path, index=False)

print("Final review tables saved:", len(final_review_inventory))
print("Manifest saved:", manifest_path)

final_review_inventory

Final review tables saved: 3
Manifest saved: c:\Users\rcolo\OneDrive\Desktop\Pennsylvania-Severe-Weather-Atlas\reports\tables\atlas_final_review_manifest.csv


,table,file,rows,columns,file_exists,values_match_after_reload
0,atlas_overview,atlas_overview_2011_2025.csv,9,3,True,True
1,atlas_impact_overview,atlas_impact_overview_2011_2025.csv,10,3,True,True
2,atlas_visual_inventory,atlas_visual_inventory.csv,11,4,True,True


## Interpreting the saved review outputs

The atlas overview, impact overview, and visual inventory were saved successfully.

All three tables reloaded with matching values. The visual inventory confirms that all 11 charts and maps exist.

# Pennsylvania Severe Weather Atlas — Final Summary

## Study scope

This atlas covers Pennsylvania from 2011 through 2025 using NOAA Storm Events records for Thunderstorm Wind, Hail, and Tornado events.

The final dataset contains **15,399 Pennsylvania records** across **67 counties**, including **564 Lehigh Valley records** from Lehigh and Northampton counties.

## Main findings

Thunderstorm Wind is the dominant event type statewide and locally:

- Pennsylvania: 12,630 wind, 2,441 hail, and 328 tornado records.
- Lehigh Valley: 464 wind, 93 hail, and 7 tornado records.
- July has the most pooled records in both regions.
- Summer contains 60.67% of Pennsylvania records and 69.33% of Lehigh Valley records.
- Pennsylvania’s highest annual total was 1,571 records in 2019.
- The Lehigh Valley’s highest annual total was 98 records in 2020.
- Allegheny County had the most statewide records, with 1,063.
- Lehigh County had more local records than Northampton County, with 313 compared with 251.

## Magnitudes and impacts

Most Pennsylvania wind records are estimated gusts, while measured gusts represent a smaller share of reports. Hail records most commonly report one-inch hail. EF0 and EF1 tornadoes account for 87.50% of Pennsylvania tornado records.

Pennsylvania records report 205 direct injuries, 10 indirect injuries, 18 direct fatalities, and 5 indirect fatalities.

Available damage estimates total $203,508,160 for property and $2,156,000 for crops statewide. Lehigh Valley property estimates total $1,691,000. Missing estimates remain unidentified, and no positive crop-damage estimates were recorded locally.

## Interpretation limits

These are reported event records, not a complete count of independent storms or a direct measure of hazard risk. NOAA records may be revised, damage estimates are incomplete and not inflation-adjusted, and the county boundaries are a 2025 reference layer applied to the study period.

The completed figures, maps, tables, manifests, and processed dataset provide a reproducible foundation for future severity, climatology, or forecasting work.